In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import os

In [5]:
# Define data transformations for data augmentation and normalization
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [7]:
#Define Data Directory
data_dir = 'dataset'

image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train','val']}

In [ ]:
#Create Data Loaders
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size = 4, shuffle=True, num_workers = 4) for x in ['train','val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train','val']}
print(dataset_sizes)

class_names = image_datasets['train'].classes
print(class_names)

In [11]:
from torchvision.models import ResNet18_Weights  # Import the weights enum

# Load pretrained ResNet-18 with updated syntax
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Freeze all layers except the last one
for name, param in model.named_parameters():
    if "fc" in name:  # Unfreeze the fully-connected layer
        param.requires_grad = True
    else:
        param.requires_grad = False

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  #most common use with classification problems
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9) #Use All Parameters #Stochastic Gradient Descent Optimizer

# Move model to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [ ]:
num_epochs = 7
for epoch in range(num_epochs):
    for phase in ['train','val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double() / dataset_sizes[phase]

        print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

print('Training Complete!')

In [17]:
#Save the Model
torch.save(model.state_dict(), 'work_type_classification_model.pth')